#### Скачайте данные
Загрузите данные, выполнив код ниже.

In [9]:
# from google.colab import files
# uploaded = files.upload()

In [79]:
import pandas as pd

data = pd.read_csv("../data/raw/NetflixShows.csv", encoding='cp437', sep=';')
del data['ratingDescription'], data['user rating size']

In [80]:
data

,title,rating,ratingLevel,release year,user rating score
0,White Chicks,PG-13,"crude and sexual humor, language and some drug...",2004,82.0
1,Lucky Number Slevin,R,"strong violence, sexual content and adult lang...",2006,NaN
2,Grey's Anatomy,TV-14,Parents strongly cautioned. May be unsuitable ...,2016,98.0
3,Prison Break,TV-14,Parents strongly cautioned. May be unsuitable ...,2008,98.0
4,How I Met Your Mother,TV-PG,Parental guidance suggested. May not be suitab...,2014,94.0
...,...,...,...,...,...
995,The BFG,PG,"for action/peril, some scary moments and brief...",2016,97.0
996,The Secret Life of Pets,PG,for action and some rude humor,2016,NaN
997,Precious Puppies,TV-G,Suitable for all ages.,2003,NaN
998,Beary Tales,TV-G,Suitable for all ages.,2013,NaN


#### Удалите из данных дубликаты.
- Почему они возникли?
- Много ли их? В каких группах их больше всего?

In [81]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   title              1000 non-null   str    
 1   rating             1000 non-null   str    
 2   ratingLevel        941 non-null    str    
 3   release year       1000 non-null   int64  
 4   user rating score  605 non-null    float64
dtypes: float64(1), int64(1), str(3)
memory usage: 109.3 KB


In [82]:
data.nunique()

title                496
rating                13
ratingLevel           99
release year          35
user rating score     42
dtype: int64

Найдем количество полных дублей, и удалим их из датасета при наличии.

In [83]:
duplicate = data.duplicated()
duplicate.sum()

np.int64(500)

500 строк в датасете - полные дубли, удалим их. 

In [84]:
data = data.drop_duplicates()
data.shape

(500, 5)

Проверим дублирование названий шоу и какие данные отличаются у одинаковых шоу.  

In [85]:
title_duplicate = data["title"].value_counts()
title_duplicate = title_duplicate[title_duplicate > 1]
title_duplicate.shape

(4,)

После удаления полных дублей осталось 4 шоу с повторяющимися названиями.
Выведем эти строки и найдем какие колонки отличаются у одинаковых шоу. 

In [86]:
title_duplicate.info()

<class 'pandas.Series'>
Index: 4 entries, Skins to Goosebumps
Series name: count
Non-Null Count  Dtype
--------------  -----
4 non-null      int64
dtypes: int64(1)
memory usage: 115.0 bytes


In [87]:
twice_show = data[data["title"].isin(title_duplicate.index)]
twice_show

,title,rating,ratingLevel,release year,user rating score
151,Skins,TV-MA,For mature audiences. May not be suitable for...,2013,NaN
167,Bordertown,TV-14,Parents strongly cautioned. May be unsuitable ...,2016,86.0
181,Skins,TV-MA,NaN,2017,NaN
449,Bordertown,TV-MA,For mature audiences. May not be suitable for...,2016,NaN
504,Star Wars: The Clone Wars,PG,"sci-fi action violence throughout, brief langu...",2008,57.0
512,Star Wars: The Clone Wars,TV-PG,Parental guidance suggested. May not be suitab...,2014,93.0
568,Goosebumps,TV-Y7,Suitable for children ages 7 and older,1998,88.0
632,Goosebumps,PG,"scary and intense creature action and images, ...",2015,90.0


У шоу Skins, Star Wars: The Clone Wars, Goosebumps отличаются год выпуска значит это разные шоу, не дубли.  
У шоу Bordertown отличается рейтинг (14+) и (18+).??

С дублями в датасете разобрались, проверим датасет на количетсво незаполненных ячеек.

In [88]:
data.isnull().sum()

title                  0
rating                 0
ratingLevel           33
release year           0
user rating score    244
dtype: int64

ratingLevel это описание рейтинговой группы. Заполним пустые значения описанием рейтинга, из заполненых ячеек того же рейтинга.  
user rating score это оценка пользователей, можно заполнить средним значением всего датасета, либо средним значением группировки по возрастному рейтингу


Нужно получить описание возрастного рейтинга (ratingLevel) для каждого уникального рейтинга (rating)

In [89]:
data.nunique()

title                496
rating                13
ratingLevel           99
release year          35
user rating score     42
dtype: int64

Так как количество уникальных ячеек в rating и ratingLevel разное, возьмем самые часто встречающиеся описания для каждого рейтинга.

In [90]:
ratingLevels = data.groupby("rating")["ratingLevel"].agg(lambda x: x.mode().iloc[0])
ratingLevels

rating
G                   General Audiences. Suitable for all ages.
NR                             This movie has not been rated.
PG          Parental guidance suggested. May not be suitab...
PG-13       For some rude and suggestive material, and for...
R           Restricted. May be inappropriate for children ...
TV-14       Parents strongly cautioned. May be unsuitable ...
TV-G                                   Suitable for all ages.
TV-MA       For mature audiences.  May not be suitable for...
TV-PG       Parental guidance suggested. May not be suitab...
TV-Y                                   Suitable for all ages.
TV-Y7                  Suitable for children ages 7 and older
TV-Y7-FV    Suitable for children ages 7 and older.  Conte...
UR          This movie has not been rated. Intended for ad...
Name: ratingLevel, dtype: str

In [91]:
data["ratingLevel"] = data["ratingLevel"].fillna(data["rating"].map(ratingLevels))

Теперь заполним пользовательскую оценку, медианой по рейтингу. 

In [92]:
UserScore = data.groupby("rating")["user rating score"].agg(lambda x: x.median())
UserScore

rating
G           70.0
NR          77.0
PG          86.0
PG-13       68.0
R           79.0
TV-14       86.0
TV-G        74.0
TV-MA       89.0
TV-PG       88.0
TV-Y        75.5
TV-Y7       74.5
TV-Y7-FV    72.0
UR           NaN
Name: user rating score, dtype: float64

Для одного рейтинга видим пустое значение, заполним медианой по всему датасету.

In [93]:
UserScore = UserScore.fillna(data["user rating score"].median())
UserScore

rating
G           70.0
NR          77.0
PG          86.0
PG-13       68.0
R           79.0
TV-14       86.0
TV-G        74.0
TV-MA       89.0
TV-PG       88.0
TV-Y        75.5
TV-Y7       74.5
TV-Y7-FV    72.0
UR          83.5
Name: user rating score, dtype: float64

In [94]:
# data["user rating score"] = data["user rating score"].fillna(data["rating"].map(UserScore))

In [95]:
data.isnull().sum()

title                  0
rating                 0
ratingLevel            0
release year           0
user rating score    244
dtype: int64

Добавим новый признак, основанный от рейтинга. Объединим признаки рейтинга в группы пользователей (Kids, Teen, Adult, Family)

In [96]:
Groups = {

    'G': 'Family',
    'TV-G': 'Family',

    'PG': 'Parental Guidance',
    'TV-PG': 'Parental Guidance',

    'TV-Y': 'Kids',
    'TV-Y7': 'Kids',
    'TV-Y7-FV': 'Kids',

    'PG-13': 'Teen',
    'TV-14': 'Teen',

    'R': 'Adult',
    'TV-MA': 'Adult',

    'NR': 'Unrated',
    'UR': 'Unrated'
}

In [97]:
data["ageGroup"] = data["rating"].map(Groups)

In [98]:
data

,title,rating,ratingLevel,release year,user rating score,ageGroup
0,White Chicks,PG-13,"crude and sexual humor, language and some drug...",2004,82.0,Teen
1,Lucky Number Slevin,R,"strong violence, sexual content and adult lang...",2006,NaN,Adult
2,Grey's Anatomy,TV-14,Parents strongly cautioned. May be unsuitable ...,2016,98.0,Teen
3,Prison Break,TV-14,Parents strongly cautioned. May be unsuitable ...,2008,98.0,Teen
4,How I Met Your Mother,TV-PG,Parental guidance suggested. May not be suitab...,2014,94.0,Parental Guidance
...,...,...,...,...,...,...
989,Russell Madness,PG,some rude humor and sports action,2015,NaN,Parental Guidance
993,Wiener Dog Internationals,G,General Audiences. Suitable for all ages.,2015,NaN,Family
994,Pup Star,G,General Audiences. Suitable for all ages.,2016,NaN,Family
997,Precious Puppies,TV-G,Suitable for all ages.,2003,NaN,Family


In [99]:
data.to_csv('../data/prepairedData.csv', sep=";", index=False)

In [100]:

external_data = pd.read_csv("../data/external/netflix_titles_nov_2019.csv", encoding='cp437', sep=',')

In [101]:
common = external_data['title'].isin(data['title']).sum()
print(common)

249


In [102]:
external_data['title'] = external_data['title'].str.lower().str.strip()
data['title'] = data['title'].str.lower().str.strip()

In [103]:
common = external_data['title'].isin(data['title']).sum()
print(common)

250


In [104]:
common = data.merge(
    external_data,
    on='title',
    how='left'
)

In [105]:
common

,title,rating_x,ratingLevel,release year,user rating score,ageGroup,show_id,director,cast,country,date_added,release_year,rating_y,duration,listed_in,description,type
0,white chicks,PG-13,"crude and sexual humor, language and some drug...",2004,82.0,Teen,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,lucky number slevin,R,"strong violence, sexual content and adult lang...",2006,NaN,Adult,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,grey's anatomy,TV-14,Parents strongly cautioned. May be unsuitable ...,2016,98.0,Teen,70140391.0,NaN,"Ellen Pompeo, Sandra Oh, Katherine Heigl, Just...",United States,NaN,2018.0,TV-14,15 Seasons,"Romantic TV Shows, TV Dramas",Intern (and eventual resident) Meredith Grey f...,TV Show
3,prison break,TV-14,Parents strongly cautioned. May be unsuitable ...,2008,98.0,Teen,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,how i met your mother,TV-PG,Parental guidance suggested. May not be suitab...,2014,94.0,Parental Guidance,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
506,russell madness,PG,some rude humor and sports action,2015,NaN,Parental Guidance,80028359.0,Robert Vince,"David Milchard, John Ratzenberger, Will Sasso,...",United States,"May 10, 2015",2015.0,PG,93 min,"Children & Family Movies, Comedies, Sports Movies",A spunky terrier named Russell with serious wr...,Movie
507,wiener dog internationals,G,General Audiences. Suitable for all ages.,2015,NaN,Family,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
508,pup star,G,General Audiences. Suitable for all ages.,2016,NaN,Family,80092857.0,Robert Vince,"Makenzie Moss, Jed Rees, David DeLuise, Carla ...",Canada,"October 29, 2016",2016.0,G,92 min,"Children & Family Movies, Comedies",After a singing pup with big dreams of stardom...,Movie
509,precious puppies,TV-G,Suitable for all ages.,2003,NaN,Family,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [106]:
external_data1 = pd.read_csv("../data/external/Netflix TV Shows and Movies.csv", encoding='cp437', sep=',')

In [107]:
external_data1['title'] = external_data1['title'].str.lower().str.strip()

In [108]:
common1 = external_data1['title'].isin(common['title']).sum()
print(common1)

149


In [109]:
common1 = common.merge(
    external_data1,
    on='title',
    how='left'
)

In [110]:
common1

,title,rating_x,ratingLevel,release year,user rating score,ageGroup,show_id,director,cast,country,...,index,id,type_y,description_y,release_year_y,age_certification,runtime,imdb_id,imdb_score,imdb_votes
0,white chicks,PG-13,"crude and sexual humor, language and some drug...",2004,82.0,Teen,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,lucky number slevin,R,"strong violence, sexual content and adult lang...",2006,NaN,Adult,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,grey's anatomy,TV-14,Parents strongly cautioned. May be unsuitable ...,2016,98.0,Teen,70140391.0,NaN,"Ellen Pompeo, Sandra Oh, Katherine Heigl, Just...",United States,...,233.0,ts21469,SHOW,Follows the personal and professional lives of...,2005.0,TV-14,49.0,tt0413573,7.6,293618.0
3,prison break,TV-14,Parents strongly cautioned. May be unsuitable ...,2008,98.0,Teen,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,how i met your mother,TV-PG,Parental guidance suggested. May not be suitab...,2014,94.0,Parental Guidance,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
512,russell madness,PG,some rude humor and sports action,2015,NaN,Parental Guidance,80028359.0,Robert Vince,"David Milchard, John Ratzenberger, Will Sasso,...",United States,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
513,wiener dog internationals,G,General Audiences. Suitable for all ages.,2015,NaN,Family,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
514,pup star,G,General Audiences. Suitable for all ages.,2016,NaN,Family,80092857.0,Robert Vince,"Makenzie Moss, Jed Rees, David DeLuise, Carla ...",Canada,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
515,precious puppies,TV-G,Suitable for all ages.,2003,NaN,Family,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [111]:
external_data2 = pd.read_csv("../data/external/movies_metadata.csv", encoding='cp437', sep=',')

C:\Users\rkart\AppData\Local\Temp\ipykernel_19376\3428401636.py:1: DtypeWarning: Columns (0: popularity) have mixed types. Specify dtype option on import or set low_memory=False.
  external_data2 = pd.read_csv("../data/external/movies_metadata.csv", encoding='cp437', sep=',')


In [112]:
external_data2['title'] = external_data2['title'].str.lower().str.strip()

In [113]:
common2 = common1.merge(
    external_data2,
    on='title',
    how='left'
)

In [114]:
common2

,title,rating_x,ratingLevel,release year,user rating score,ageGroup,show_id,director,cast,country,...,production_countries,release_date,revenue,runtime_y,spoken_languages,status,tagline,video,vote_average,vote_count
0,white chicks,PG-13,"crude and sexual humor, language and some drug...",2004,82.0,Teen,NaN,NaN,NaN,NaN,...,"[{'iso_3166_1': 'US', 'name': 'United States o...",2004-06-23,113086475.0,109.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,They're going deep undercover.,False,6.3,704.0
1,lucky number slevin,R,"strong violence, sexual content and adult lang...",2006,NaN,Adult,NaN,NaN,NaN,NaN,...,"[{'iso_3166_1': 'DE', 'name': 'Germany'}, {'is...",2006-02-24,56308881.0,110.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Wrong Time. Wrong Place. Wrong Number.,False,7.4,1356.0
2,grey's anatomy,TV-14,Parents strongly cautioned. May be unsuitable ...,2016,98.0,Teen,70140391.0,NaN,"Ellen Pompeo, Sandra Oh, Katherine Heigl, Just...",United States,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,prison break,TV-14,Parents strongly cautioned. May be unsuitable ...,2008,98.0,Teen,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,how i met your mother,TV-PG,Parental guidance suggested. May not be suitab...,2014,94.0,Parental Guidance,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
591,russell madness,PG,some rude humor and sports action,2015,NaN,Parental Guidance,80028359.0,Robert Vince,"David Milchard, John Ratzenberger, Will Sasso,...",United States,...,"[{'iso_3166_1': 'US', 'name': 'United States o...",2015-02-21,0.0,92.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,The strongest tag team is family.,False,5.1,7.0
592,wiener dog internationals,G,General Audiences. Suitable for all ages.,2015,NaN,Family,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
593,pup star,G,General Audiences. Suitable for all ages.,2016,NaN,Family,80092857.0,Robert Vince,"Makenzie Moss, Jed Rees, David DeLuise, Carla ...",Canada,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
594,precious puppies,TV-G,Suitable for all ages.,2003,NaN,Family,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [116]:
"show_id"
"rating_y"
"description_x"
"index"
"id_x"
"description_y"
"release_year_y"
"age_certification"
"imdb_id_x"
"adult"
"belongs_to_collection"
"homepage"
"id_y"
"imdb_id_y"
"overview"
"poster_path"
"production_countries"
"spoken_languages"
"status"
"tagline"
"video"
"vote_average"
"vote_count"

'vote_count'

In [117]:
df = common2.drop(columns=["show_id", "rating_y", "description_x", "index", "id_x", "description_y", "release_year_y", "age_certification", "imdb_id_x",
                           "adult", "belongs_to_collection", "homepage", "id_y", "imdb_id_y", "overview", "poster_path", "production_countries", "spoken_languages",
                           "status", "tagline", "video", "vote_average", "vote_count", "runtime_x", "original_language", "original_title"])

In [118]:
df

,title,rating_x,ratingLevel,release year,user rating score,ageGroup,director,cast,country,date_added,...,type_y,imdb_score,imdb_votes,budget,genres,popularity,production_companies,release_date,revenue,runtime_y
0,white chicks,PG-13,"crude and sexual humor, language and some drug...",2004,82.0,Teen,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,37000000,"[{'id': 35, 'name': 'Comedy'}]",7.723341,"[{'name': 'Columbia Pictures', 'id': 5}, {'nam...",2004-06-23,113086475.0,109.0
1,lucky number slevin,R,"strong violence, sexual content and adult lang...",2006,NaN,Adult,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,27000000,"[{'id': 18, 'name': 'Drama'}, {'id': 53, 'name...",11.15092,"[{'name': 'The Weinstein Company', 'id': 308},...",2006-02-24,56308881.0,110.0
2,grey's anatomy,TV-14,Parents strongly cautioned. May be unsuitable ...,2016,98.0,Teen,NaN,"Ellen Pompeo, Sandra Oh, Katherine Heigl, Just...",United States,NaN,...,SHOW,7.6,293618.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,prison break,TV-14,Parents strongly cautioned. May be unsuitable ...,2008,98.0,Teen,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,how i met your mother,TV-PG,Parental guidance suggested. May not be suitab...,2014,94.0,Parental Guidance,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
591,russell madness,PG,some rude humor and sports action,2015,NaN,Parental Guidance,Robert Vince,"David Milchard, John Ratzenberger, Will Sasso,...",United States,"May 10, 2015",...,NaN,NaN,NaN,0,"[{'id': 35, 'name': 'Comedy'}]",0.65691,"[{'name': 'Air Bud Entertainment', 'id': 48861}]",2015-02-21,0.0,92.0
592,wiener dog internationals,G,General Audiences. Suitable for all ages.,2015,NaN,Family,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
593,pup star,G,General Audiences. Suitable for all ages.,2016,NaN,Family,Robert Vince,"Makenzie Moss, Jed Rees, David DeLuise, Carla ...",Canada,"October 29, 2016",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
594,precious puppies,TV-G,Suitable for all ages.,2003,NaN,Family,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [120]:
import ast

In [121]:
df['genres'] = df['genres'].apply(
    lambda x: [g['name'] for g in ast.literal_eval(x)]
    if pd.notna(x) else x
)

In [122]:
df['production_companies'] = df['production_companies'].apply(
    lambda x: [g['name'] for g in ast.literal_eval(x)]
    if pd.notna(x) else x
)

In [124]:
df['type'] = df['type_x'].combine_first(df['type_y'])

In [125]:
df = df.drop(columns=['type_x', 'type_y'])

In [126]:
df.to_csv('../data/prepairedexternalData.csv', sep=";", index=False)